In [1]:
import json
import numpy as np

# ─────────────────────────────────────────
# STEP 1: Gini Impurity
# ─────────────────────────────────────────
def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

# ─────────────────────────────────────────
# STEP 2: Best Split Finder
# ─────────────────────────────────────────
def find_best_split(X, y):
    n_samples, n_features = X.shape
    parent_gini = gini_impurity(y)
    best_gain      = -1
    best_feature   = None
    best_threshold = None

    for feature_idx in range(n_features):
        thresholds = np.unique(X[:, feature_idx])
        for threshold in thresholds:
            left_mask  = X[:, feature_idx] <= threshold
            right_mask = ~left_mask
            y_left     = y[left_mask]
            y_right    = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            weight_left   = len(y_left)  / n_samples
            weight_right  = len(y_right) / n_samples
            weighted_gini = (weight_left  * gini_impurity(y_left) +
                             weight_right * gini_impurity(y_right))
            info_gain = parent_gini - weighted_gini

            if info_gain > best_gain:
                best_gain      = info_gain
                best_feature   = feature_idx
                best_threshold = threshold

    return best_feature, best_threshold, best_gain

# ─────────────────────────────────────────
# STEP 3: Recursive Tree Builder
# ─────────────────────────────────────────
def build_tree(X, y, depth=0, max_depth=3):
    if gini_impurity(y) == 0:
        return {"leaf": True, "prediction": int(y[0])}

    if depth >= max_depth:
        majority_class = int(np.bincount(y).argmax())
        return {"leaf": True, "prediction": majority_class}

    best_feature, best_threshold, best_gain = find_best_split(X, y)

    if best_gain <= 0:
        majority_class = int(np.bincount(y).argmax())
        return {"leaf": True, "prediction": majority_class}

    left_mask  = X[:, best_feature] <= best_threshold
    right_mask = ~left_mask

    left_subtree  = build_tree(X[left_mask],  y[left_mask],  depth + 1, max_depth)
    right_subtree = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth)

    return {
        "leaf"      : False,
        "feature"   : int(best_feature),
        "threshold" : float(best_threshold),
        "gain"      : round(float(best_gain), 4),
        "left"      : left_subtree,
        "right"     : right_subtree
    }

# ─────────────────────────────────────────
# STEP 4: Data + Build + Print
# ─────────────────────────────────────────
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

X = np.array([
    [2.5, 1.0],
    [1.0, 3.5],
    [3.0, 2.0],
    [0.5, 2.5],
    [4.0, 0.5],
    [1.5, 4.0],
])
y = np.array([0, 1, 0, 1, 0, 1])

tree = build_tree(X, y, max_depth=3)
print(json.dumps(tree, indent=2, cls=NumpyEncoder))

{
  "leaf": false,
  "feature": 0,
  "threshold": 1.5,
  "gain": 0.5,
  "left": {
    "leaf": true,
    "prediction": 1
  },
  "right": {
    "leaf": true,
    "prediction": 0
  }
}
